# 対訳コーパスを取得して中身を確認する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/26_machine_translation/parallel_corpus.ipynb)

NLP クックブック 26-01 ／ 機械翻訳

機械翻訳の実験用に、日英の対訳データを手に入れて、品質や長さの分布を確認したい。

In [1]:
!pip install -q datasets

In [2]:
from datasets import load_dataset

ds = load_dataset("Helsinki-NLP/opus-100", "en-ja")     # OPUS から抽出した 100 言語の対訳 (日英は 100 万文対)
print(ds)
for ex in ds["train"].select(range(5)):
    print(ex["translation"]["ja"], "|", ex["translation"]["en"])

DatasetDict({
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
    train: Dataset({
        features: ['translation'],
        num_rows: 1000000
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})
- ラウール - ラウールに ヴィンセント・ハンナだ | Yeah, Vincent Hanna.
いま地下に居ます 他の2人と一緒に誘拐されたんです！ | I'm being held in a basement. I've been abducted with two other girls.
動いたよ！ | It works!
何があったか突き止めたい | I'm just trying to find out what happened here.
無事か？ | You okay?


In [3]:
import pandas as pd

sample = ds["train"].shuffle(seed=0).select(range(20000))
df = pd.DataFrame(sample["translation"])
df["len_ja"], df["len_en"] = df["ja"].str.len(), df["en"].str.split().str.len()
df["ratio"] = df["len_ja"] / df["len_en"].clip(lower=1)
print(df[["len_ja", "len_en", "ratio"]].describe().round(1))

# 対訳として怪しいものを機械的に除く: 空、長すぎ、長さの比が極端、原文と訳文が同じ、日本語を含まない
ok = ((df["len_ja"] > 0) & (df["len_en"] > 0) & (df["len_en"] <= 100) & df["ratio"].between(0.5, 8)
      & (df["ja"] != df["en"]) & df["ja"].str.contains(r"[ぁ-んァ-ヶ一-龥]"))
print(f"残る割合: {ok.mean():.1%}")
print(df[~ok].sample(5, random_state=0)[["ja", "en"]].to_string(index=False))

        len_ja   len_en    ratio
count  20000.0  20000.0  20000.0
mean      15.6      7.3      2.6
std       18.6      6.8      2.8
min        1.0      1.0      0.1
25%        7.0      4.0      1.5
50%       12.0      6.0      2.0
75%       18.0      9.0      2.9
max      741.0    172.0    141.0
残る割合: 97.1%
                                                                                                                                           ja                en
                                                                                                                         ああ、そうだ では、幾つか写真を送信する         Yes, lam.
                                                                                                らせん銀河 (NGC 2685, おおぐま座)object name (optional)             Helix
                                                                                                                         ジャックが正しい たぶん目の前だけを −     Jack's right.
                                                   

- OPUS の日英は映画字幕 (OpenSubtitles) 由来が多く、口語的で短く、ノイズや意訳も多い。データの出どころと分野を必ず確認する。
- 日英の定番データ: JParaCrawl（Web、大規模）、ASPEC（科学技術論文、要申請）、KFTT（京都関連の Wikipedia）、JESC（字幕）、WMT の news タスク。
- 長さの比によるフィルタは最低限の処理。さらに、言語判定、文埋め込みの類似度（LaBSE など）によるフィルタが有効。